In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import cooler
import bioframe

import cooltools


In [ ]:
binsize=100
resolution=100

wt= cooler.Cooler('PATH_TO/A.mcool::/resolutions/'+str(resolution))
cool= cooler.Cooler('PATH_TO/B.mcool::/resolutions/'+str(resolution))

In [ ]:
# just checking
print(cool.chromnames)

In [ ]:
#load feature list
cenless = pd.read_csv('PATH_TO/features.csv')

#load all genome features
arms = pd.read_csv('Documents/pileups/whole_genome.csv') 


In [ ]:
#create stacks
wt_stack = cooltools.pileup(wt, cenless, view_df = arms, flank=5000, min_diag=0)
stack = cooltools.pileup(cool, cenless, view_df = arms, flank=5000, min_diag=0)

# Mirror reflect snippets when the feature is on the opposite strand
wt_mask = np.array(cenless.strand == '-', dtype=bool)
wt_stack[wt_mask, :, :] = wt_stack[wt_mask, ::-1, ::-1]

# Mirror reflect snippets when the feature is on the opposite strand
mask = np.array(cenless.strand == '-', dtype=bool)
stack[mask, :, :] = stack[mask, ::-1, ::-1]

# Aggregate. Note that some pixels might be converted to NaNs after IC, thus we aggregate by nanmean:
wt_mtx = np.nanmean(wt_stack, axis=0) 

# Aggregate. Note that some pixels might be converted to NaNs after IC, thus we aggregate by nanmean:
mtx = np.nanmean(stack, axis=0) 

In [ ]:
#calculate vmin and max to be the best scale based on the wildtype data
#all samples are then plotted on the same scale
log_mtx = np.log10(wt_mtx[np.isfinite(wt_mtx) & (wt_mtx > 0)])

vmin = np.percentile(log_mtx, 1)
vmax = np.percentile(log_mtx, 99)


In [ ]:
#plot the pileup
flank=5000
plt.imshow(
    np.log10(mtx),
    cmap='viridis_r',
    vmin = vmin,
    vmax = vmax,
    interpolation='none')

plt.colorbar(label = 'log10 mean ICed MicroC')
ticks_pixels = np.linspace(0, flank*2//resolution,5)
ticks_bp = ((ticks_pixels)*resolution)
plt.xticks(ticks_pixels, ticks_bp)
plt.yticks(ticks_pixels, ticks_bp)
plt.xlabel('relative position, bp')
plt.ylabel('relative position, bp')
plt.rcParams['pdf.fonttype'] = 'truetype'
plt.rcParams['svg.fonttype'] = 'none' # No text as paths. Assume font installed.
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['text.usetex'] = False

plt.savefig("output_file_name.svg")
plt.show()